# Chapter 5 — What the track cannot see

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/05_beyond_the_track.ipynb)

**Goal.** Chapter 4 fitted the *path* of each galaxy in the [Si/Fe]–[Fe/H] plane. A path is not a history: it does
not say how big the galaxy is, nor how many stars it formed at each point along the path. This chapter shows, with
models whose answer we know, what the abundance track misses, and how the **stellar mass at each metallicity**
(the metallicity distribution in solar masses) recovers it.

**You will learn**
1. why abundance ratios are blind to a galaxy's mass;
2. how two galaxies with almost the same track can have formed their stars at different metallicities;
3. how adding the mass distribution to a fit recovers a history that abundances alone confuse;
4. what the gas left today adds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

## 5.1 The model

We use Chapter 2's numpy model (instantaneous recycling, constant yields, exponential DTD), with H21's delayed-τ
inflow, $\dot M_{\rm in} = (M_i/\tau_i)(t/\tau_i)\,e^{-t/\tau_i}$, and an initial gas mass $M_0$. It is copied here so
that the chapter runs on its own, and it is fast enough to run many times.

In [ ]:
YIELD = {'mg': (1.098e-3, 3.791e-5), 'si': (1.175e-3, 3.411e-4), 'fe': (1.165e-3, 1.860e-3)}   # Chapter 2, Z = 0.002
LOGEPS = {'fe': 7.47, 'si': 7.54, 'mg': 7.55}
A = {'fe': 56, 'si': 28, 'mg': 24}
X_H = 0.73


def dtd_exp(tau, t_d=0.15, tau_ia=1.5):
    return np.where(tau >= t_d, np.exp(-(tau - t_d) / tau_ia) / tau_ia, 0.0)


def galaxy(tau_i, tau_star, eta, M0, Mi=1e10, t_end=12.0, dt=0.01, r=0.4):
    """Chapter 2's model with delayed-tau inflow of total mass Mi [Msun] on time scale tau_i [Gyr].
    Returns t [Gyr], gas mass [Msun], SFR [Msun/Gyr], [Fe/H], [Si/Fe]."""
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    Mg = np.zeros(n + 1)
    Mg[0] = M0
    M = {el: np.zeros(n + 1) for el in YIELD}
    sfr = np.zeros(n + 1)
    f = dtd_exp(t) * dt
    for i in range(n):
        sfr[i] = Mg[i] / tau_star
        ia = np.dot(sfr[:i + 1], f[i::-1])
        loss = (1 + eta - r) * sfr[i]
        for el, (y_cc, y_ia) in YIELD.items():
            z = M[el][i] / Mg[i] if Mg[i] > 0 else 0.0
            M[el][i + 1] = M[el][i] + dt * (y_cc * sfr[i] + y_ia * ia - loss * z)
        Mg[i + 1] = Mg[i] + dt * (Mi / tau_i * (t[i] / tau_i) * np.exp(-t[i] / tau_i) - loss)
    sfr[n] = Mg[n] / tau_star
    with np.errstate(divide='ignore', invalid='ignore'):
        feh = np.log10(M['fe'] / Mg / A['fe'] / X_H) - (LOGEPS['fe'] - 12)
        sife = np.log10((M['si'] / A['si']) / (M['fe'] / A['fe'])) - (LOGEPS['si'] - LOGEPS['fe'])
    return dict(t=t, mgas=Mg, sfr=sfr, feh=feh, sife=sife, dt=dt)

## 5.2 Abundances cannot see size

Every term in the model is proportional to a mass: the inflow, the gas, the star formation, the outflow, the metals.
Multiply all of them by the same factor and the equations are unchanged, so every *ratio*, [Fe/H] and [Si/Fe]
included, is unchanged. Check it: shrink a galaxy tenfold.

In [ ]:
big = galaxy(tau_i=4.0, tau_star=10.0, eta=5.0, M0=3e9, Mi=1e10)
small = galaxy(tau_i=4.0, tau_star=10.0, eta=5.0, M0=3e8, Mi=1e9)
ok = big['t'] > 0
print(f"max |Δ[Fe/H]| = {np.max(np.abs(big['feh'] - small['feh'])[ok]):.1e},  "
      f"max |Δ[Si/Fe]| = {np.max(np.abs(big['sife'] - small['sife'])[ok]):.1e},  "
      f"stars formed: {big['sfr'].sum() * big['dt']:.3e} vs {small['sfr'].sum() * small['dt']:.3e} Msun")
assert np.allclose(big['feh'][ok], small['feh'][ok], atol=1e-10)          # identity: the track ignores size
assert np.allclose(big['sife'][ok], small['sife'][ok], atol=1e-10)
assert np.allclose(small['sfr'], 0.1 * big['sfr'], rtol=1e-10)           # ... while every mass scales by 0.1

The tracks agree to rounding error while every mass differs tenfold. This is why H21 could give all five galaxies
the same inflow mass ($6\times10^{10}\,M_\odot$): the fit to abundances cannot tell. A measured **total stellar
mass** fixes this one number. If a model with inflow mass $M_i$ forms $M_\star^{\rm model}$ of stars, the galaxy
needs $M_i \times M_\star^{\rm obs}/M_\star^{\rm model}$, with no other change.

## 5.3 Same track, different histories

Mass is not the only thing a track hides. Here are two galaxies with the same outflow ($\eta$ = 5) and the same
inflow ($\tau_i$ = 8 Gyr):
- **A** forms stars with $\tau_\star$ = 10 Gyr and starts with no gas;
- **B** forms stars half as efficiently ($\tau_\star$ = 20 Gyr) but starts with $3\times10^9\,M_\odot$ of gas.

In [ ]:
EDGES = np.r_[-9.0, np.arange(-2.0, 0.01, 0.25), 9.0]       # [Fe/H] bins: below -2, 0.25 dex up to 0, above 0
CENTRES = np.r_[-2.125, (EDGES[1:-2] + EDGES[2:-1]) / 2, 0.125]


def mass_per_bin(m):
    """Stellar mass formed in each [Fe/H] bin [Msun]: SFR x dt of every step, binned by the gas [Fe/H] at that step."""
    ok = np.isfinite(m['feh'])
    return np.histogram(m['feh'][ok], EDGES, weights=m['sfr'][ok] * m['dt'])[0]


def track_on(m, feh):
    """[Si/Fe] of the model track at the given [Fe/H] values (on its rising part)."""
    ok = np.isfinite(m['feh'])
    return np.interp(feh, np.maximum.accumulate(m['feh'][ok]), m['sife'][ok])


ga = galaxy(tau_i=8.0, tau_star=10.0, eta=5.0, M0=0.0)
gb = galaxy(tau_i=8.0, tau_star=20.0, eta=5.0, M0=3e9)
x = np.linspace(-1.8, -0.8, 41)
d_track = np.max(np.abs(track_on(ga, x) - track_on(gb, x)))
fa, fb = mass_per_bin(ga), mass_per_bin(gb)
below = lambda f: f[:5].sum() / f.sum()                     # fraction of the stellar mass below [Fe/H] = -1
gas_ratio = lambda m: m['mgas'][-1] / (m['sfr'][:-1].sum() * m['dt'])
print(f"largest track difference for -1.8 < [Fe/H] < -0.8: {d_track:.3f} dex;  final [Fe/H]: {ga['feh'][-1]:.2f}, {gb['feh'][-1]:.2f}")
print(f"mass fraction below [Fe/H] = -1: A {below(fa):.3f}, B {below(fb):.3f};  gas today / stars formed: A {gas_ratio(ga):.2f}, B {gas_ratio(gb):.2f}")
assert d_track < 0.015 and abs(ga['feh'][-1] - gb['feh'][-1]) < 0.01     # the tracks agree ...
assert below(fb) / below(fa) > 4                                           # ... their histories do not

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for m, f, col, lab in ((ga, fa, '#2a78d6', r'A: $\tau_*$ = 10 Gyr, no initial gas'),
                       (gb, fb, '#eb6834', r'B: $\tau_*$ = 20 Gyr, $M_0 = 3\times10^9\,M_\odot$')):
    axes[0].plot(m['feh'], m['sife'], color=col, lw=2, label=lab)
    axes[1].plot(m['t'], m['sfr'] / 1e9, color=col, lw=2, label=lab)
    axes[2].step(CENTRES, f / f.sum(), where='mid', color=col, lw=2, label=lab)
axes[0].errorbar(-0.7, 0.2, xerr=0.03, yerr=0.03, color='k', capsize=3)
axes[0].text(-0.65, 0.2, 'typical APOGEE error', va='center', fontsize=8)
axes[0].set(xlim=(-2.5, 0), ylim=(-0.15, 0.3), xlabel='[Fe/H]', ylabel='[Si/Fe]')
axes[1].set(xlabel='time [Gyr]', ylabel=r'SFR [$M_\odot\,$yr$^{-1}$]')
axes[2].set(xlim=(-2.3, 0.1), xlabel='[Fe/H]', ylabel='fraction of stellar mass per 0.25-dex bin')
for ax in axes:
    ax.legend(frameon=False, fontsize=8)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 5.1.** Two model galaxies with the same inflow ($\tau_i$ = 8 Gyr, $M_i = 10^{10}\,M_\odot$) and outflow
($\eta$ = 5): A (blue; $\tau_\star$ = 10 Gyr, no initial gas) and B (orange; $\tau_\star$ = 20 Gyr, initial gas
$3\times10^9\,M_\odot$). Left: [Si/Fe] against [Fe/H], with a typical APOGEE error bar (±0.03 dex). Middle: star-formation
rate against time. Right: fraction of each galaxy's stellar mass formed in each 0.25-dex bin of [Fe/H] (the lowest
bin collects everything below −2).

The tracks differ by at most 0.013 dex over −1.8 < [Fe/H] < −0.8, less than half an APOGEE error bar, and end at
the same [Fe/H]. Yet B formed 17% of its stars below [Fe/H] = −1 and A only 4%. B starts with a full gas
reservoir and forms stars at once, while the gas is still metal-poor; A must first accumulate gas from the inflow.
The lower efficiency of B slows its enrichment just enough to keep the tracks together. The two also end with
different amounts of gas (printed above).

## 5.4 A fit with and without the mass distribution

Does this matter for a fit? Build a small grid of twelve galaxies: $\tau_i$ = 2, 4 or 8 Gyr, $\tau_\star$ = 10 or
20 Gyr, $M_0$ = 0 or $3\times10^9\,M_\odot$, all with $\eta$ = 5. Take B as the truth and make 20 mock data sets,
each with
- 800 stars drawn along B's track, with probability proportional to its star formation, plus Gaussian errors of
  0.03 dex in both [Fe/H] and [Si/Fe];
- B's mass fractions per [Fe/H] bin, with errors of 10% of each fraction plus 0.005 (we assume such a measurement
  exists; making it from a survey requires correcting for the survey's selection, beyond this tutorial).

Fit each mock twice: with the abundance score of Chapter 4 alone, and with that score plus a $\chi^2$ of the mass
fractions.

In [ ]:
def chi2_track(points, errors, track):
    """Chapter 4's score: sum of |d|^4 / sum_k d_k^2 sigma_k^2, d = star minus nearest point on the track."""
    v = np.all(np.isfinite(track), axis=1)
    seg = v[:-1] & v[1:]
    a, b = track[:-1][seg], track[1:][seg]
    ab = b - a
    ab2 = np.einsum('ij,ij->i', ab, ab)
    chi = np.empty(len(points))
    for s in range(0, len(points), 128):
        p, e = points[s:s + 128], errors[s:s + 128]
        d = p[:, None, :] - a
        f = np.clip(np.divide(np.einsum('nmi,mi->nm', d, ab), ab2, out=np.zeros(d.shape[:2]), where=ab2 > 0), 0, 1)
        q = a + f[:, :, None] * ab
        d2 = np.sum((p[:, None, :] - q) ** 2, axis=2)
        k = np.argmin(d2, axis=1)
        r = p - q[np.arange(len(k)), k]
        den = np.sum(r ** 2 * e ** 2, axis=1)
        chi[s:s + 128] = np.divide(d2[np.arange(len(k)), k] ** 2, den, out=np.zeros(len(k)), where=den > 0)
    return float(chi.sum())


GRID = {p: galaxy(tau_i=p[0], tau_star=p[1], eta=5.0, M0=p[2]) for p in product((2.0, 4.0, 8.0), (10.0, 20.0), (0.0, 3e9))}
truth = (8.0, 20.0, 3e9)
tracks = {p: np.c_[m['feh'], m['sife']][1:] for p, m in GRID.items()}      # drop t = 0, where [Fe/H] = -inf
fracs = {p: mass_per_bin(m) / mass_per_bin(m).sum() for p, m in GRID.items()}

rng = np.random.default_rng(1)
hits = {'abundances only': 0, 'abundances + mass distribution': 0}
wrong = []
for trial in range(20):
    m = GRID[truth]
    ok = np.isfinite(m['feh']) & (m['t'] > 0)
    w = m['sfr'] * ok / np.sum(m['sfr'] * ok)
    born = rng.choice(len(w), 800, p=w)                       # birth steps, weighted by star formation
    err = np.full((800, 2), 0.03)
    pts = np.c_[m['feh'][born], m['sife'][born]] + rng.normal(size=err.shape) * err
    sig = 0.1 * fracs[truth] + 0.005
    f_obs = fracs[truth] + rng.normal(size=sig.size) * sig
    c_ab = {p: chi2_track(pts, err, tr) for p, tr in tracks.items()}
    c_mass = {p: np.sum((f_obs - f) ** 2 / sig ** 2) for p, f in fracs.items()}
    best_ab = min(c_ab, key=c_ab.get)
    best_joint = min(c_ab, key=lambda p: c_ab[p] + c_mass[p])
    hits['abundances only'] += best_ab == truth
    hits['abundances + mass distribution'] += best_joint == truth
    if best_ab != truth:
        wrong.append(best_ab)
print(hits)
print('abundance-only fits that missed chose (tau_i, tau_star, M0):', sorted(set(wrong)))
assert hits['abundances + mass distribution'] == 20 and hits['abundances only'] == 8       # recovery (seeded)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(list(hits), [v / 20 for v in hits.values()], color=['#888888', '#2a78d6'])
ax.set(xlim=(0, 1), xlabel='fraction of 20 mocks in which the true model is the best fit')
ax.grid(color='0.9', lw=0.6, axis='x')
fig.tight_layout()

**Figure 5.2.** Fraction of 20 mock data sets (truth: galaxy B) in which the fit picks the true model out of the
twelve, using the abundance score alone (grey) or adding the mass fractions per [Fe/H] bin (blue).

With abundances alone the fit finds the truth in only 8 of the 20 mocks; every miss chooses the galaxy with the
faster inflow ($\tau_i$ = 4 Gyr, otherwise identical), whose track is nearly the same. With the mass distribution it
finds the truth every time. In these mocks the abundance errors are exactly known; in real data, with $\chi^2/N$ of
4–7 (Chapter 4), the abundance score discriminates even less, and the mass distribution matters more.

## 5.5 The gas left today

The ratio of the gas left at the end to the stars formed is, like the abundances, independent of the galaxy's size
(§5.2), but it depends on the history: a galaxy with inefficient star formation or with a late inflow is left with
much gas.

In [ ]:
print('gas today / stars formed over the twelve-model grid:')
for p, m in GRID.items():
    print(f'  tau_i = {p[0]:.0f}, tau_star = {p[1]:.0f}, M0 = {p[2]:.0e}:  {gas_ratio(m):.2f}')
ratios = np.array([gas_ratio(m) for m in GRID.values()])
assert abs(gas_ratio(big) - gas_ratio(small)) < 1e-10                    # identity: independent of size
assert abs(ratios.min() - 0.16) < 0.005 and abs(ratios.max() - 2.80) < 0.005   # but it varies with the history

Over this small grid the ratio runs from 0.16 to 2.8 (printed above). A measured gas mass is therefore a
third independent constraint: it pins down how much gas the galaxy has *not* yet turned into stars or lost, which
depends on the efficiency, the outflow and the inflow history together. Abundance ratios, mass per [Fe/H], total
mass and gas mass constrain different combinations of parameters, and a model that must match all four has far
less freedom than one fitted to the track alone.

## Summary

- Abundance ratios are blind to size: scaling all masses leaves the track unchanged to rounding error. A total
  stellar mass fixes the inflow mass.
- Two galaxies can share a track to 0.013 dex yet form 4% and 17% of their stars below [Fe/H] = −1.
- In mocks, adding the mass distribution to the abundance fit raises the recovery of the true history to 20/20.
- The gas left today, relative to the stars formed, is size-independent and history-dependent: one more constraint.

## Exercises

1. Make the mock abundance errors 0.06 dex instead of 0.03 and repeat §5.4. How often does each fit recover the truth?
2. Add a fourth constraint to the fit: B's gas ratio with a 20% error. Does it alone separate B from its confusers?
3. H21's best models form $10^9$–$10^{10}\,M_\odot$ of stars (Chapter 3, Exercise 3). Look up the stellar masses of
   the LMC, SMC and Fornax (McConnachie 2012, https://ui.adsabs.harvard.edu/abs/2012AJ....144....4M) and compute the
   inflow mass each galaxy would need.